# Model evaluation workbench: messy deals to auditable evaluation

This notebook generates synthetic commercial data with the repository's real generator, degrades the export, restores its schema, proves recovery against known truth, and runs the real XGBoost/Random Forest evaluation with held-out metrics, cross-validation, residuals, SHAP importance, and serialized bundles.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run this notebook from the repository root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "notebooks" / "data"
ARTIFACT_DIR = ROOT / "notebooks" / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository: {ROOT.name}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
import pandas as pd

from model_evaluation_workbench.data import generate_deal_dataset

canonical = generate_deal_dataset(rows=500, seed=42)
messy = canonical.copy().astype(object)
messy["annual_spend"] = messy["annual_spend"].map(lambda value: f" ${value:,.2f} ")
messy["audience_reach"] = messy["audience_reach"].map(lambda value: f"{int(value):,}")
for column in ["deal_type", "market", "vertical"]:
    messy[column] = messy[column].map(lambda value: f" {str(value).upper()} ")
messy["legacy_note"] = "imported from spreadsheet"

invalid = messy.iloc[[0]].copy()
invalid["annual_spend"] = "not available"
messy = pd.concat([messy, messy.iloc[:8], invalid], ignore_index=True)
messy = messy.sample(frac=1, random_state=42).reset_index(drop=True)
messy_path = DATA_DIR / "messy_deals.csv"
messy.to_csv(messy_path, index=False)
print(
    {
        "canonical_rows": len(canonical),
        "messy_rows": len(messy),
        "file": str(messy_path.relative_to(ROOT)),
    }
)
messy.head(3)

In [ ]:
cleaned = messy.drop(columns=["legacy_note"]).copy()
cleaned["annual_spend"] = pd.to_numeric(
    cleaned["annual_spend"].astype(str).str.replace(r"[$,\s]", "", regex=True),
    errors="coerce",
)
cleaned["audience_reach"] = pd.to_numeric(
    cleaned["audience_reach"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
for column in ["deal_type", "market", "vertical"]:
    cleaned[column] = cleaned[column].astype(str).str.strip().str.lower()
cleaned = cleaned.dropna().drop_duplicates().reset_index(drop=True)
for column in ["audience_reach", "activation_channels", "term_months"]:
    cleaned[column] = cleaned[column].astype(int)
cleaned = cleaned[canonical.columns]

sort_columns = list(canonical.columns)
actual = cleaned.sort_values(sort_columns).reset_index(drop=True)
expected = canonical.sort_values(sort_columns).reset_index(drop=True)
pd.testing.assert_frame_equal(actual, expected, check_dtype=False)
clean_path = DATA_DIR / "cleaned_deals.csv"
cleaned.to_csv(clean_path, index=False)
print(
    f"Recovered all {len(cleaned)} canonical rows and removed {len(messy) - len(cleaned)} bad/duplicate rows."
)

In [ ]:
import json

from model_evaluation_workbench.evaluation import evaluate_models

output = ARTIFACT_DIR / "evaluation"
metrics = evaluate_models(cleaned, output, estimators=50, cv_folds=3, seed=42, compute_shap=True)
report = json.loads((output / "evaluation_report.json").read_text())

required = {
    "evaluation_report.json",
    "held_out_predictions.csv",
    "xgboost_bundle.joblib",
    "random_forest_bundle.joblib",
    "xgboost_shap_importance.csv",
    "random_forest_shap_importance.csv",
}
assert required.issubset({path.name for path in output.iterdir()})
assert report["train_rows"] + report["held_out_rows"] == len(cleaned)
{name: vars(result) for name, result in metrics.items()}

In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(completed.stdout)
print(completed.stderr)
assert completed.returncode == 0, "Repository tests failed"
print("VALIDATED: the repository test suite passes in this UV environment.")